# SportRadar Tennis Data Pipeline (Robust Edition)
This notebook automates the process of fetching tennis data from the Sportradar API, cleaning it, and transforming it into tabular DataFrames.

### Version Notes:
- **Enhanced Diagnostics:** Added detailed error messages for 403 (Forbidden) and other access issues.
- **Flexible Structure:** Improved parsing logic for rankings to handle multiple Sportradar JSON variations.
- **SQL-Ready Types:** Explicitly converts numeric columns for seamless SQL integration.

In [52]:
import requests
import pandas as pd
import time
import json

# --- CONFIGURATION ---
API_KEY = "f5zLPaq7vFTRpMVziqArzh13pCLD8KZMxDzKdFEn"
# Set ACCESS_LEVEL to "trial" or "production"
ACCESS_LEVEL = "trial" 
BASE_URL = f"https://api.sportradar.com/tennis/{ACCESS_LEVEL}/v3/en"
HEADERS = {"accept": "application/json"}

## 1. Data Fetching
We fetch data from three main endpoints. If you see a **403 error**, please verify on the Sportradar Developer Portal that your key is authorized for **Tennis v3**.

In [55]:
def fetch_sportradar_data(endpoint):
    url = f"{BASE_URL}/{endpoint}.json?api_key={API_KEY}"
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 403:
            print(f"[Error 403] Access Forbidden for {endpoint}.")
            print(f"CAUSE: Your key is likely NOT authorized for Tennis v3 {ACCESS_LEVEL} access.")
            print("FIX: Check your Sportradar Portal and ensure 'Tennis API v3' is active.")
        elif response.status_code == 429:
            print("[Error 429] Rate Limit Exceeded. Please wait before retrying.")
        else:
            print(f"[Error {response.status_code}] for {endpoint}: {response.text[:200]}")
    except Exception as e:
        print(f"Connection Error for {endpoint}: {e}")
    return None

print("Starting data extraction...")

# Fetch Competitions
competitions_raw = fetch_sportradar_data("competitions")
time.sleep(1.2) # Trial rate limit is usually 1 call per second

# Fetch Complexes
complexes_raw = fetch_sportradar_data("complexes")
time.sleep(1.2)

# Fetch Doubles Rankings
rankings_raw = fetch_sportradar_data("double_competitors_rankings")

import os
import json

raw_dir = 'data/raw_data'
if not os.path.exists(raw_dir):
    os.makedirs(raw_dir)

if competitions_raw:
    with open(os.path.join(raw_dir, 'live_competitions.json'), 'w') as f:
        json.dump(competitions_raw, f)
if complexes_raw:
    with open(os.path.join(raw_dir, 'live_complexes.json'), 'w') as f:
        json.dump(complexes_raw, f)
if rankings_raw:
    with open(os.path.join(raw_dir, 'live_rankings.json'), 'w') as f:
        json.dump(rankings_raw, f)

print("\nExtraction phase finished and JSONs saved to data/raw_data/")

Starting data extraction...

Extraction phase finished and JSONs saved to data/raw_data/


## 2. Data Transformation & Cleaning
This section converts the raw JSON into normalized DataFrames. We use `pd.to_numeric` to ensure values are ready for SQL mathematical operations.

In [58]:
### 2.1 Competitions & Categories
if competitions_raw and 'competitions' in competitions_raw:
    raw_comps = competitions_raw['competitions']
    
    # Extract Categories table
    categories_list = []
    for comp in raw_comps:
        cat = comp.get('category')
        if cat:
            categories_list.append({'category_id': cat.get('id'), 'category_name': cat.get('name')})
    categories_df = pd.DataFrame(categories_list).drop_duplicates().reset_index(drop=True)
    
    # Extract Competitions table
    competitions_list = [{
        'competition_id': c.get('id'),
        'competition_name': c.get('name'),
        'parent_id': c.get('parent_id'),
        'type': c.get('type'),
        'gender': c.get('gender'),
        'category_id': c.get('category', {}).get('id')
    } for c in raw_comps]
    competitions_df = pd.DataFrame(competitions_list)
    
    print("Parsed Competitions and Categories successfully.")
else:
    print("Skipping Competitions parse: Data missing or fetch failed.")

Parsed Competitions and Categories successfully.


In [60]:
### 2.2 Complexes & Venues
if complexes_raw and 'complexes' in complexes_raw:
    raw_complexes = complexes_raw['complexes']
    
    complexes_list = []
    venues_list = []
    
    for comp in raw_complexes:
        complexes_list.append({'complex_id': comp.get('id'), 'complex_name': comp.get('name')})
        
        for venue in comp.get('venues', []):
            venues_list.append({
                'venue_id': venue.get('id'),
                'venue_name': venue.get('name'),
                'city_name': venue.get('city_name'),
                'country_name': venue.get('country_name'),
                'country_code': venue.get('country_code'),
                'timezone': venue.get('timezone'),
                'complex_id': comp.get('id')
            })
    
    complexes_df = pd.DataFrame(complexes_list)
    venues_df = pd.DataFrame(venues_list)
    print("Parsed Complexes and Venues successfully.")
else:
    print("Skipping Complexes parse: Data missing or fetch failed.")

Parsed Complexes and Venues successfully.


In [62]:
### 2.3 Competitors & Rankings
if rankings_raw and 'rankings' in rankings_raw:
    # Robust check for structure: results can be in nested rankings[0] or direct
    rank_data = rankings_raw['rankings']
    all_rankings = []
    
    if isinstance(rank_data, list) and len(rank_data) > 0:
        if 'competitor_rankings' in rank_data[0]:
            all_rankings = rank_data[0]['competitor_rankings']
        else:
            all_rankings = rank_data # Direct list of rank objects
    
    competitors_list = []
    rankings_list = []
    
    for entry in all_rankings:
        competitor = entry.get('competitor', {})
        cid = competitor.get('id')
        
        competitors_list.append({
            'competitor_id': cid,
            'name': competitor.get('name'),
            'country': competitor.get('country'),
            'country_code': competitor.get('country_code'),
            'abbreviation': competitor.get('abbreviation')
        })
        
        rankings_list.append({
            'rank': entry.get('rank'),
            'movement': entry.get('movement'),
            'points': entry.get('points'),
            'competitions_played': entry.get('competitions_played'),
            'competitor_id': cid
        })
    
    competitors_df = pd.DataFrame(competitors_list).drop_duplicates(subset=['competitor_id'])
    rankings_df = pd.DataFrame(rankings_list)
    
    # Enforce numeric types for SQL readiness
    for col in ['rank', 'movement', 'points', 'competitions_played']:
        rankings_df[col] = pd.to_numeric(rankings_df[col], errors='coerce').fillna(0).astype(int)
        
    print("Parsed Competitors and Rankings successfully.")
else:
    print("Skipping Rankings parse: Data missing or fetch failed.")

Parsed Competitors and Rankings successfully.


## 3. Previews
If the tables above were parsed successfully, you can view the top 5 rows of each below.

In [65]:
try:
    print("\n--- CATEGORIES ---")
    display(categories_df.head())
    print("\n--- COMPETITIONS ---")
    display(competitions_df.head())
    print("\n--- COMPLEXES ---")
    display(complexes_df.head())
    print("\n--- VENUES ---")
    display(venues_df.head())
    print("\n--- COMPETITIORS ---")
    display(competitors_df.head())
    print("\n--- RANKINGS ---")
    display(rankings_df.head())
except NameError:
    print("No DataFrames to display. Please verify your API Key and Access Level at the top.")


--- CATEGORIES ---


,category_id,category_name
0,sr:category:181,Hopman Cup
1,sr:category:3,ATP
2,sr:category:72,Challenger
3,sr:category:6,WTA
4,sr:category:76,Davis Cup



--- COMPETITIONS ---


,competition_id,competition_name,parent_id,type,gender,category_id
0,sr:competition:620,Hopman Cup,None,mixed,mixed,sr:category:181
1,sr:competition:660,World Team Cup,None,mixed,men,sr:category:3
2,sr:competition:990,ATP Challenger Tour Finals,sr:competition:6239,singles,men,sr:category:72
3,sr:competition:1207,Championship International Series,None,singles,women,sr:category:6
4,sr:competition:2100,Davis Cup,None,mixed,men,sr:category:76



--- COMPLEXES ---


...

,venue_id,venue_name,city_name,country_name,country_code,timezone,complex_id
0,sr:venue:70045,Cancha Central,Santiago,CHILE,CHL,America/Santiago,sr:complex:705
1,sr:venue:74856,Centre Court,Seville,SPAIN,ESP,Europe/Madrid,sr:complex:1078
2,sr:venue:74858,Court One,Seville,SPAIN,ESP,Europe/Madrid,sr:complex:1078
3,sr:venue:1500,CENTER COURT,Saint Petersburg,RUSSIAN FEDERATION,RUS,Europe/Moscow,sr:complex:1495
4,sr:venue:62153,TC Dynamo,Saint Petersburg,RUSSIAN FEDERATION,RUS,Europe/Moscow,sr:complex:1495



--- COMPETITIORS ---


,competitor_id,name,country,country_code,abbreviation
0,sr:competitor:16160,"Zeballos, Horacio",Argentina,ARG,ZEB
1,sr:competitor:15568,"Granollers, Marcel",Spain,ESP,GRA
2,sr:competitor:53785,"Skupski, Neal",Great Britain,GBR,SKU
3,sr:competitor:59131,"Glasspool, Lloyd",Great Britain,GBR,GLA
4,sr:competitor:108099,"Cash, Julian",Great Britain,GBR,CAS



--- RANKINGS ---


,rank,movement,points,competitions_played,competitor_id
0,1,1,8610,14,sr:competitor:16160
1,2,1,8520,15,sr:competitor:15568
2,3,-2,8430,25,sr:competitor:53785
3,4,0,7280,22,sr:competitor:59131
4,5,0,7130,23,sr:competitor:108099


## 4. Export to CSV
Finally, we export the cleaned DataFrames to the `data/` directory so they can be loaded into SQL or BI tools.

In [68]:
import os

processed_dir = 'data/processed_data'
if not os.path.exists(processed_dir):
    os.makedirs(processed_dir)

try:
    categories_df.to_csv(os.path.join(processed_dir, 'categories.csv'), index=False)
    competitions_df.to_csv(os.path.join(processed_dir, 'competitions.csv'), index=False)
    complexes_df.to_csv(os.path.join(processed_dir, 'complexes.csv'), index=False)
    venues_df.to_csv(os.path.join(processed_dir, 'venues.csv'), index=False)
    competitors_df.to_csv(os.path.join(processed_dir, 'competitors.csv'), index=False)
    rankings_df.to_csv(os.path.join(processed_dir, 'rankings.csv'), index=False)
    print("Successfully exported 6 cleaned DataFrames to CSV in the data/processed_data/ folder.")
except NameError as e:
    print(f"Error: {e}. Please ensure you run the data fetching and cleaning cells above first.")

Successfully exported 6 cleaned DataFrames to CSV in the data/processed_data/ folder.
